# Credit Scoring — Data Preprocessing

## Objective

The goal of this stage is to prepare the credit scoring
dataset for machine learning.

The preprocessing pipeline will include:

- Data leakage prevention
- Train/test split
- Missing value imputation
- Categorical feature encoding
- Numerical feature scaling

The prepared data will be used to train a Logistic Regression model.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Dataset/lending_club_clean.csv")

df.shape

(150000, 20)

## 2. Train-Test Split

Before preprocessing the data, we split the dataset into training and testing sets.

The training set will be used to train machine learning models, while the test set will be used to evaluate their performance on unseen data.

We use an 80/20 split:

- 80% for training.
- 20% for testing.

Since the target variable is imbalanced, we use stratified splitting to preserve the original class distribution in both sets.

A fixed random state ensures reproducibility.

In [4]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["default"])
y = df["default"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

X_train: (120000, 19)
X_test: (30000, 19)

Training target distribution:
default
0    79.818333
1    20.181667
Name: proportion, dtype: float64

Testing target distribution:
default
0    79.816667
1    20.183333
Name: proportion, dtype: float64


In [6]:
print("Training features:", X_train.shape)
print("Training targets:", y_train.shape)

print("Testing features:", X_test.shape)
print("Testing targets:", y_test.shape)

Training features: (120000, 19)
Training targets: (120000,)
Testing features: (30000, 19)
Testing targets: (30000,)


## 3. Missing Value Imputation

Missing values must be handled before training the model.

To prevent data leakage, imputation statistics are calculated
using only the training dataset.

The same statistics are then applied to both training
and testing datasets.

We use `SimpleImputer` from Scikit-learn with the median strategy.

In [7]:
from sklearn.impute import SimpleImputer

# Example training data
train_data = [[30000], [40000], [50000], [None]]

# Example testing data
test_data = [[60000], [None]]

# Create imputer
imputer = SimpleImputer(strategy="median")

# Learn median from training data
imputer.fit(train_data)

# Fill missing values
train_filled = imputer.transform(train_data)
test_filled = imputer.transform(test_data)

print("Train:")
print(train_filled)

print("\nTest:")
print(test_filled)

Train:
[[30000.]
 [40000.]
 [50000.]
 [40000.]]

Test:
[[60000.]
 [40000.]]


### Applying Imputation to the Dataset

We apply median imputation to numerical features with missing values.

The imputer is fitted only on the training data to prevent data leakage.

The same learned median values are then used to transform both training and testing datasets.

At this stage, we preserve the original datasets and create separate processed copies.

In [8]:
# Features containing missing values
missing_features = ["emp_length", "dti", "revol_util"]

# Create copies to preserve original data
X_train_filled = X_train.copy()
X_test_filled = X_test.copy()

# Create imputer
imputer = SimpleImputer(strategy="median")

# Learn medians from Train and fill missing values
X_train_filled[missing_features] = imputer.fit_transform(
    X_train[missing_features]
)

# Fill Test using the same medians
X_test_filled[missing_features] = imputer.transform(
    X_test[missing_features]
)

In [9]:
print("Missing values in Train:")
print(X_train_filled.isnull().sum().sum())

print("\nMissing values in Test:")
print(X_test_filled.isnull().sum().sum())

print("\nLearned medians:")
print(dict(zip(missing_features, imputer.statistics_)))

Missing values in Train:
0

Missing values in Test:
0

Learned medians:
{'emp_length': np.float64(6.0), 'dti': np.float64(17.57), 'revol_util': np.float64(52.2)}


## 4. Categorical Feature Analysis

Before encoding categorical features, we inspect their unique
values and determine whether they are nominal or ordinal.

Nominal features have no natural order, while ordinal features
represent ordered categories.

The encoding strategy will be selected based on feature semantics.

In [10]:
categorical_features = [
    "grade",
    "sub_grade",
    "home_ownership",
    "verification_status",
    "purpose"
]

for col in categorical_features:
    print(f"\n{col}:")
    print(sorted(X_train_filled[col].unique()))


grade:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

sub_grade:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34)]

home_ownership:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

verification_status:
[np.int64(0), np.int64(1), np.int64(2)]

purpose:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]


### One-Hot Encoding

Nominal categorical features are encoded using One-Hot Encoding.

This transformation creates separate binary columns for each category,
avoiding artificial numerical relationships between categories.

The encoder is fitted on the training dataset and then applied
to both training and testing datasets.

In [11]:
from sklearn.preprocessing import OneHotEncoder

nominal_features = [
    "home_ownership",
    "verification_status",
    "purpose"
]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_encoded = encoder.fit_transform(
    X_train_filled[nominal_features]
)

X_test_encoded = encoder.transform(
    X_test_filled[nominal_features]
)

print("Train:", X_train_encoded.shape)
print("Test:", X_test_encoded.shape)

Train: (120000, 23)
Test: (30000, 23)


In [12]:
encoded_columns = encoder.get_feature_names_out(nominal_features)

print("Number of encoded columns:", len(encoded_columns))
print("\nEncoded columns:")
print(encoded_columns)

Number of encoded columns: 23

Encoded columns:
['home_ownership_0' 'home_ownership_1' 'home_ownership_2'
 'home_ownership_3' 'home_ownership_4' 'home_ownership_5'
 'verification_status_0' 'verification_status_1' 'verification_status_2'
 'purpose_0' 'purpose_1' 'purpose_2' 'purpose_3' 'purpose_4' 'purpose_5'
 'purpose_6' 'purpose_7' 'purpose_8' 'purpose_9' 'purpose_10' 'purpose_11'
 'purpose_12' 'purpose_13']


In [13]:
encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoded_columns
)

encoded_df.head()

,home_ownership_0,home_ownership_1,home_ownership_2,home_ownership_3,home_ownership_4,home_ownership_5,verification_status_0,verification_status_1,verification_status_2,purpose_0,...,purpose_4,purpose_5,purpose_6,purpose_7,purpose_8,purpose_9,purpose_10,purpose_11,purpose_12,purpose_13
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Feature Scaling

Numerical features have different scales and ranges.

We use StandardScaler to standardize numerical features.

The scaler learns the mean and standard deviation from the training data only.

The same parameters are then applied to the testing data.

Categorical features are not scaled.

In [14]:
numeric_features = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "emp_length",
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc"
]

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train_filled.copy()
X_test_scaled = X_test_filled.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(
    X_train_filled[numeric_features]
)

X_test_scaled[numeric_features] = scaler.transform(
    X_test_filled[numeric_features]
)

In [16]:
print("Before scaling:")
print(X_train_filled[numeric_features].describe().loc[["mean", "std"]])

print("\nAfter scaling:")
print(X_train_scaled[numeric_features].describe().loc[["mean", "std"]])

Before scaling:
         loan_amnt       term   int_rate  installment  emp_length  \
mean  14391.565625  41.779600  13.247804   437.346984    5.958875   
std    8700.429204  10.261944   4.756425   260.895363    3.580595   

        annual_inc        dti  delinq_2yrs  inq_last_6mths   open_acc  \
mean  75995.833864  18.254718     0.317283        0.661892  11.585108   
std   67716.518045  11.496828     0.868958        0.947505   5.463728   

       pub_rec     revol_bal  revol_util  total_acc  
mean  0.215925  16216.933683   51.863096  25.035708  
std   0.606510  21944.107717   24.518341  12.026684  

After scaling:
         loan_amnt          term      int_rate   installment    emp_length  \
mean  9.716672e-17 -1.634840e-16 -5.512627e-17 -8.443616e-17  5.352755e-17   
std   1.000004e+00  1.000004e+00  1.000004e+00  1.000004e+00  1.000004e+00   

        annual_inc           dti   delinq_2yrs  inq_last_6mths      open_acc  \
mean  1.004234e-16 -7.555438e-17  1.918465e-17    6.560678e-17 